# Logistic Regression with Gradient Descent (1D)

**Contents**
1. Data
2. Model
3. Loss and gradients
4. Training
5. Evaluation

Cells marked **Example** can simply be run. Cells marked **Exercise** contain `# TODO` gaps for you to complete.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---
## 1. Data

### Example 1.1: A binary classification dataset

$x$ = hours revised, $y = 1$ if the student passed, $0$ otherwise.

In [ ]:
rng = np.random.default_rng(seed=0)
n = 200

x = rng.uniform(0, 10, n)
p_true = 1 / (1 + np.exp(-1.2 * (x - 5)))
y = rng.binomial(1, p_true)

print("Class counts:", np.bincount(y))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x, y, alpha=0.5)
ax.set_xlabel("Hours revised, $x$")
ax.set_ylabel("Passed, $y$")
ax.set_yticks([0, 1])
plt.show()

---
## 2. Model

### Example 2.1: The sigmoid function

$$\sigma(z) = \frac{1}{1 + e^{-z}} \qquad\qquad f(x) = \sigma(wx + b)$$

In [ ]:
def sigmoid(z):
    """Map any real number to the interval (0, 1)."""
    return 1 / (1 + np.exp(-z))


def predict_prob(x, w, b):
    """Return the predicted probability that y = 1."""
    return sigmoid(w * x + b)

In [ ]:
z = np.linspace(-8, 8, 200)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(z, sigmoid(z))
ax.axhline(0.5, color="grey", lw=0.8, ls="--")
ax.set_xlabel("$z$")
ax.set_ylabel(r"$\sigma(z)$")
plt.show()

### Exercise 1: From probabilities to labels

Complete the function so that it returns `1` where the predicted probability is at least the threshold (=D), and `0` otherwise.

In [ ]:
def predict(x, w, b, threshold=0.5):
    """Return predicted class labels (0 or 1)."""
    # TODO: get the predicted probabilities
    # TODO: return them as integer labels using the threshold

In [ ]:
# Check your answer (should print True)
labels = predict(np.array([0.0, 4.0, 10.0]), w=1.2, b=-6.0)
print(np.array_equal(labels, np.array([0, 0, 1])))

---
## 3. Loss and Gradients

### Example 3.1: Binary cross-entropy

$$J(w, b) = -\frac{1}{m}\sum_{i=1}^{m} \Big[ y^{(i)} \log({f}^{(i)}) + (1 - y^{(i)})\log(1 - {f}^{(i)}) \Big]$$

Squared error is not used here: it is not convex for this model, and this loss punishes confident wrong answers much more heavily.

In [ ]:
def binary_cross_entropy(y_true, f_pred, eps=1e-12):
    """Return the binary cross-entropy loss."""
    f_pred = np.clip(f_pred, eps, 1 - eps)   # avoid log(0)
    return -np.mean(y_true * np.log(f_pred) + (1 - y_true) * np.log(1 - f_pred))

### Example 3.2: Gradients

As we showed in the lecture, these have exactly the same form as in linear regression:

$$\frac{\partial J}{\partial w} = \frac{1}{m}\sum_{i=1}^{m} \,({f}^{(i)}-y^{(i)})\,x^{(i)}
\qquad\qquad
\frac{\partial J}{\partial b} = \frac{1}{m}\sum_{i=1}^{m} ({f}^{(i)}-y^{(i)})$$

In [ ]:
def gradients(x, y, w, b):
    """Return the gradients (dJ/dw, dJ/db) of the cross-entropy cost."""
    error = y - predict_prob(x, w, b)
    dw = -np.mean(x * error)
    db = -np.mean(error)
    return dw, db

In [ ]:
w_guess, b_guess = 0.0, 0.0

print("Cost:  ", binary_cross_entropy(y, predict_prob(x, w_guess, b_guess)))
print("dJ/dw: ", gradients(x, y, w_guess, b_guess)[0])
print("dJ/db: ", gradients(x, y, w_guess, b_guess)[1])

### Exercise 2: The loss curve

Fix $b = -6$ and plot the cost for values of $w$ between $-1$ and $4$. Is it convex?

In [ ]:
w_values = np.linspace(-1, 4, 200)
costs = []
    
for w in w_values:
    # TODO: compute the cost for this w (with b = -6) and append it to costs


fig, ax = plt.subplots(figsize=(6, 4))
# TODO: plot costs against w_values, with axis labels


---
## 4. Training

### Example 4.1: Gradient descent

$$w \leftarrow w - \alpha\,\frac{\partial J}{\partial w} \qquad\qquad b \leftarrow b - \alpha\,\frac{\partial J}{\partial b}$$

In [ ]:
def gradient_descent(x, y, alpha=0.5, n_iters=2000, w=0.0, b=0.0):
    """Fit w and b by gradient descent. Returns w, b and the cost history."""
    history = []
    for _ in range(n_iters):
        dw, db = gradients(x, y, w, b)
        w = w - alpha * dw
        b = b - alpha * db
        history.append(binary_cross_entropy(y, predict_prob(x, w, b)))
    return w, b, history

In [ ]:
w_fit, b_fit, history = gradient_descent(x, y, alpha=0.5, n_iters=2000)

print(f"w = {w_fit:.3f}   (``true'' value: 1.2)")
print(f"b = {b_fit:.3f}  (``true'' value: -6.0)")
print(f"Final cost = {history[-1]:.4f}")

The fit does not recover the generating values exactly: with 200 noisy labels, the best-fitting curve is a bit flatter.

### Example 4.2: Loss curve and fitted model

In [ ]:
x_line = np.linspace(0, 10, 200)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(history)
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Cost")

axes[1].scatter(x, y, alpha=0.4)
axes[1].plot(x_line, predict_prob(x_line, w_fit, b_fit), color="red", label="Fitted model")
axes[1].axhline(0.5, color="grey", lw=0.8, ls="--")
axes[1].set_xlabel("Hours revised, $x$")
axes[1].set_ylabel("$P(y = 1)$")
axes[1].set_title("Fitted model")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 5. Evaluation

### Example 5.1: Predicted labels and accuracy

In [ ]:
y_prob = predict_prob(x, w_fit, b_fit)
y_pred = (y_prob >= 0.5).astype(int)

accuracy = np.mean(y_pred == y)
print(f"Accuracy: {accuracy:.3f}")

### Exercise 3: Precision, recall and F1

Count the four outcomes, then compute the metrics.

| | Predicted 0 | Predicted 1 |
|---|---|---|
| **Actual 0** | TN | FP |
| **Actual 1** | FN | TP |

$$\text{precision} = \frac{TP}{TP + FP} \qquad
\text{recall} = \frac{TP}{TP + FN} \qquad
F_1 = \frac{2 \times \text{precision} \times \text{recall}}{\text{precision} + \text{recall}}$$

In [ ]:
# TODO: count each of the four outcomes (hint: np.sum((y == 1) & (y_pred == 1)))
tp = 
tn =
fp =
fn =

print(f"TP = {tp}, TN = {tn}, FP = {fp}, FN = {fn}")

In [ ]:
# TODO: compute each metric from tp, tn, fp, fn
accuracy = 
precision = 
recall = 
f1 = 

print(f"Accuracy:  {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1 score:  {f1:.3f}")

In [ ]:
# Check your answer (should print True)
print(np.isclose(accuracy, np.mean(y_pred == y)) and tp + tn + fp + fn == len(y))

### Exercise 4: Visualise the confusion matrix

Build the 2 × 2 matrix and display it with `imshow`, writing each count inside its cell.

In [ ]:
# TODO: build a 2x2 array with rows = actual (0, 1) and columns = predicted (0, 1)
cm = np.array([[tn, ...],
               [..., ...]])

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.imshow(cm, cmap="Blues")

for i in range(2):
    for j in range(2):
        # TODO: write the count cm[i, j] at position (j, i) using ax.text

ax.set_xticks([0, 1], ["Predicted 0", "Predicted 1"])
ax.set_yticks([0, 1], ["Actual 0", "Actual 1"])
ax.set_title("Confusion matrix")
plt.tight_layout()
plt.show()

Try a threshold of 0.8 instead of 0.5 in Example 5.1 and rerun Exercises 3 and 4.
Which metric improves, and which gets worse?

## Technical - try and re-train your logistic regression model (predict_prob) using a mean squared error cost function instead of cross-entropy. Plot J as a function of w and/or b. Plot J as a function in gradient descent iteration - what happens?